# <span style="color:green"> ANÁLISE EXPLORATÓRIA INDIVIDUAL</span> #

## <span style="color:green"> PACOTES UTILIZADOS </span> ##

In [18]:
import pandas as pd
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io
import base64
from pathlib import Path
import io
import base64
from matplotlib.ticker import FuncFormatter

## <span style="color:green"> IMPORTE DE BASE DE DADOS E DIRECIONAMENTO DOS ESTUDOS ADA </span> ##

In [19]:

dataset_prime = pd.read_parquet(
    "../../dados/dataset_final/otimizado/otimizacao_final/dataset_prime.parquet"
)

print(dataset_prime.shape)

for i, coluna in enumerate(dataset_prime.columns, start=1):
    print(f"{i}. {coluna}")

    

(1852394, 22)
1. NID_ALPHA
2. TRANS_NUM_CARD_FEWF
3. RECIVE_LOC_FEWF
4. RECIVE_CATEGORY_OHEWI
5. TRANS_VALUE
6. SEND_GENDER_BE
7. SEND_LAT_REGISTER
8. SEND_LONG_REGISTER
9. SEND_POP_REGISTER
10. SEND_JOB_FEWF
11. RECIVE_LAT
12. RECIVE_LONG
13. TRANS_DAY
14. TRANS_WEEK_OHEWI
15. TRANS_YEAR_BE
16. TRANS_MONTH_SEN
17. TRANS_MONTH_COS
18. TRANS_HOUR_SEN
19. TRANS_HOUR_COS
20. SEND_NAME_FEWF
21. SEND_AGE
22. TARGET_OMEGA


## <span style="color:green"> BINARY ENCODING </span> ##

### <span style="color:white"> TARGET_OMEGA </span> ###

In [20]:
# ============================================================
# 1. CAMINHOS
# ============================================================

RAIZ_PROJETO = Path("/projeto_tcc_2026")

CAMINHO_DATASET = (
    RAIZ_PROJETO
    / "dados"
    / "dataset_final"
    / "otimizado"
    / "otimizacao_final"
    / "dataset_prime.parquet"
)

DIRETORIO_RESULTADOS = (
    RAIZ_PROJETO
    / "resultados"
    / "analise_exploratoria_individual_features"
    / "binary_encoding"
    / "target_omega"
)

CAMINHO_HTML = (
    DIRETORIO_RESULTADOS
    / "analise_target_omega.html"
)


# ============================================================
# 2. VERIFICAÇÃO DOS CAMINHOS
# ============================================================

if not CAMINHO_DATASET.exists():
    raise FileNotFoundError(
        f"Dataset não encontrado:\n{CAMINHO_DATASET}"
    )

if not DIRETORIO_RESULTADOS.exists():
    raise FileNotFoundError(
        f"Pasta de resultados não encontrada:\n"
        f"{DIRETORIO_RESULTADOS}"
    )


# ============================================================
# 3. CARREGAMENTO APENAS DA FEATURE TARGET_OMEGA
# ============================================================

dataset_target = pd.read_parquet(
    CAMINHO_DATASET,
    columns=["TARGET_OMEGA"]
)

target = dataset_target["TARGET_OMEGA"]


# ============================================================
# 4. VALIDAÇÃO DA FEATURE
# ============================================================

if target.isna().any():
    raise ValueError(
        "TARGET_OMEGA possui valores ausentes."
    )

valores_encontrados = set(
    target.unique()
)

if not valores_encontrados.issubset({0, 1}):
    raise ValueError(
        "TARGET_OMEGA possui valores diferentes de 0 e 1: "
        f"{valores_encontrados}"
    )


# ============================================================
# 5. QUANTIDADE ABSOLUTA DE CADA CLASSE
# ============================================================

contagem = (
    target
    .value_counts()
    .reindex([0, 1], fill_value=0)
)

nao_fraude = int(
    contagem.loc[0]
)

fraude = int(
    contagem.loc[1]
)

total = int(
    contagem.sum()
)


# ============================================================
# 6. PROPORÇÃO PERCENTUAL DE CADA CLASSE
# ============================================================

percentual_nao_fraude = (
    nao_fraude / total
) * 100

percentual_fraude = (
    fraude / total
) * 100


# ============================================================
# 7. RAZÃO DE DESBALANCEAMENTO
# ============================================================

if fraude > 0:

    razao_desbalanceamento = (
        nao_fraude / fraude
    )

else:

    razao_desbalanceamento = np.inf


# ============================================================
# 8. IMBALANCE RATIO (IR)
# ============================================================

classe_majoritaria = int(
    contagem.max()
)

classe_minoritaria = int(
    contagem.min()
)

if classe_minoritaria > 0:

    imbalance_ratio = (
        classe_majoritaria
        / classe_minoritaria
    )

else:

    imbalance_ratio = np.inf


# ============================================================
# 9. ENTROPIA DE SHANNON
# ============================================================

proporcoes = (
    contagem / total
)

entropia = -sum(
    p * np.log2(p)
    for p in proporcoes
    if p > 0
)


# ============================================================
# 10. FUNÇÃO PARA:
#     - SALVAR PNG EM 600 DPI
#     - CONVERTER FIGURA PARA BASE64 PARA O HTML
# ============================================================

def salvar_figura_e_converter_base64(
    fig,
    nome_arquivo,
    diretorio
):

    # --------------------------------------------------------
    # Salva PNG em alta resolução
    # --------------------------------------------------------

    caminho_png = (
        diretorio
        / f"{nome_arquivo}.png"
    )

    fig.savefig(
        caminho_png,
        format="png",
        dpi=600,
        bbox_inches="tight"
    )

    # --------------------------------------------------------
    # Gera versão Base64 para incorporar no HTML
    # --------------------------------------------------------

    buffer = io.BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=200,
        bbox_inches="tight"
    )

    buffer.seek(0)

    imagem_base64 = base64.b64encode(
        buffer.read()
    ).decode("utf-8")

    buffer.close()

    plt.close(fig)

    return imagem_base64


# ============================================================
# 11. GRÁFICO 1
#     DISTRIBUIÇÃO DAS CLASSES EM ESCALA LOGARÍTMICA
# ============================================================

rotulos = [
    "Não fraude (0)",
    "Fraude (1)"
]

valores = [
    nao_fraude,
    fraude
]

fig, ax = plt.subplots(
    figsize=(8, 6)
)

barras = ax.bar(
    rotulos,
    valores
)

ax.set_yscale(
    "log"
)

ax.set_title(
    "Distribuição da TARGET_OMEGA em escala logarítmica"
)

ax.set_xlabel(
    "Classe"
)

ax.set_ylabel(
    "Quantidade de transações"
)

ax.grid(
    axis="y",
    alpha=0.3
)

# ------------------------------------------------------------
# Valor absoluto acima de cada barra
# Sem separador de milhar
# ------------------------------------------------------------

for barra, valor in zip(
    barras,
    valores
):

    ax.text(
        barra.get_x()
        + barra.get_width() / 2,
        valor,
        str(valor),
        ha="center",
        va="bottom"
    )


fig.tight_layout()


grafico_log_base64 = (
    salvar_figura_e_converter_base64(
        fig,
        "target_omega_distribuicao_log",
        DIRETORIO_RESULTADOS
    )
)


# ============================================================
# 12. GRÁFICO 2
#     FRAUDES ACUMULADAS POR ORDEM DAS TRANSAÇÕES
# ============================================================

# Eixo X:
# 1 = primeira transação
# 2 = segunda transação
# 3 = terceira transação
# ...

# Eixo Y:
# quantidade acumulada de fraudes


target_array = (
    target.to_numpy()
)


# ------------------------------------------------------------
# Localiza as posições onde TARGET_OMEGA = 1
# ------------------------------------------------------------

posicoes_fraude = (
    np.flatnonzero(
        target_array == 1
    )
    + 1
)


# ------------------------------------------------------------
# Cria os pontos necessários para a curva acumulada
# ------------------------------------------------------------

if len(posicoes_fraude) > 0:

    x_acumulado = np.concatenate(
        (
            [1],
            posicoes_fraude,
            [total]
        )
    )

    y_acumulado = np.concatenate(
        (
            [0],
            np.arange(
                1,
                len(posicoes_fraude) + 1
            ),
            [len(posicoes_fraude)]
        )
    )

else:

    x_acumulado = np.array(
        [1, total]
    )

    y_acumulado = np.array(
        [0, 0]
    )


# ------------------------------------------------------------
# Construção do gráfico
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(12, 6)
)

ax.step(
    x_acumulado,
    y_acumulado,
    where="post"
)

ax.set_title(
    "Fraudes acumuladas por ordem das transações"
)

ax.set_xlabel(
    "Ordem da transação"
)

ax.set_ylabel(
    "Quantidade acumulada de fraudes"
)

ax.set_xlim(
    1,
    total
)

ax.set_ylim(
    bottom=0
)

ax.grid(
    alpha=0.3
)


# ------------------------------------------------------------
# Formatação dos eixos
# Sem separador de milhar
# ------------------------------------------------------------

ax.xaxis.set_major_formatter(
    FuncFormatter(
        lambda x, pos: str(int(x))
    )
)

ax.yaxis.set_major_formatter(
    FuncFormatter(
        lambda y, pos: str(int(y))
    )
)


fig.tight_layout()


grafico_acumulado_base64 = (
    salvar_figura_e_converter_base64(
        fig,
        "target_omega_fraudes_acumuladas",
        DIRETORIO_RESULTADOS
    )
)


# ============================================================
# 13. CRIAÇÃO DO HTML
# ============================================================

conteudo_html = f"""
<!DOCTYPE html>

<html lang="pt-BR">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Análise Exploratória Individual - TARGET_OMEGA
</title>

<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1100px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 25px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 10px;
    text-align: center;
}}

.resultado {{
    font-size: 18px;
    font-weight: bold;
}}

.grafico {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 35px;
}}

.grafico img {{
    max-width: 100%;
    height: auto;
}}

.resumo {{
    margin-bottom: 40px;
}}

</style>

</head>

<body>


<h1>
Análise Exploratória Individual — TARGET_OMEGA
</h1>


<p>

A variável <strong>TARGET_OMEGA</strong>
corresponde à variável-alvo binária do conjunto de dados.

</p>


<ul>

<li>
<strong>0:</strong>
transação não fraudulenta
</li>

<li>
<strong>1:</strong>
transação fraudulenta
</li>

</ul>


<h2>
1. Quantidade absoluta de cada classe
</h2>


<table>

<thead>

<tr>
<th>Classe</th>
<th>Significado</th>
<th>Quantidade</th>
</tr>

</thead>


<tbody>

<tr>
<td>0</td>
<td>Não fraude</td>
<td>{nao_fraude}</td>
</tr>

<tr>
<td>1</td>
<td>Fraude</td>
<td>{fraude}</td>
</tr>

<tr>
<td><strong>Total</strong></td>
<td>-</td>
<td><strong>{total}</strong></td>
</tr>

</tbody>

</table>


<h2>
2. Proporção percentual de cada classe
</h2>


<table>

<thead>

<tr>
<th>Classe</th>
<th>Significado</th>
<th>Percentual</th>
</tr>

</thead>


<tbody>

<tr>
<td>0</td>
<td>Não fraude</td>
<td>{percentual_nao_fraude:.6f}%</td>
</tr>

<tr>
<td>1</td>
<td>Fraude</td>
<td>{percentual_fraude:.6f}%</td>
</tr>

</tbody>

</table>


<h2>
3. Razão de desbalanceamento
</h2>


<p class="resultado">

{razao_desbalanceamento:.2f}:1

</p>


<p>

Existe aproximadamente

<strong>

1 transação fraudulenta para cada
{razao_desbalanceamento:.2f}
transações não fraudulentas.

</strong>

</p>


<h2>
4. Imbalance Ratio (IR)
</h2>


<p class="resultado">

IR = {imbalance_ratio:.4f}

</p>


<p>

O Imbalance Ratio representa a razão entre
a quantidade de observações da classe majoritária
e a quantidade de observações da classe minoritária.

</p>


<p>

Quanto maior o valor do IR,
maior é o desbalanceamento entre as classes.

</p>


<h2>
5. Entropia de Shannon
</h2>


<p class="resultado">

Entropia = {entropia:.6f} bits

</p>


<p>

Para uma variável binária,
a Entropia de Shannon varia entre 0 e 1.

</p>


<ul>

<li>

Valores próximos de <strong>0</strong>
indicam maior concentração das observações
em uma única classe.

</li>

<li>

Valores próximos de <strong>1</strong>
indicam maior equilíbrio entre as classes.

</li>

</ul>


<h2>
6. Distribuição das classes em escala logarítmica
</h2>


<p>

A escala logarítmica facilita a comparação visual
entre as duas classes quando existe uma grande diferença
entre suas frequências absolutas.

</p>


<div class="grafico">

<img
src="data:image/png;base64,{grafico_log_base64}"
alt="Distribuição da TARGET_OMEGA em escala logarítmica"
>

</div>


<h2>
7. Fraudes acumuladas por ordem das transações
</h2>


<p>

O eixo X representa a posição sequencial das transações
no conjunto de dados.

A posição 1 corresponde à primeira transação,
a posição 2 à segunda transação,
a posição 3 à terceira transação
e assim sucessivamente.

</p>


<p>

O eixo Y representa a quantidade acumulada de fraudes.

Sempre que uma observação apresenta
<strong>TARGET_OMEGA = 1</strong>,
o valor acumulado é incrementado em uma unidade.

Quando a observação apresenta
<strong>TARGET_OMEGA = 0</strong>,
o valor acumulado permanece constante.

</p>


<div class="grafico">

<img
src="data:image/png;base64,{grafico_acumulado_base64}"
alt="Fraudes acumuladas por ordem das transações"
>

</div>


<h2>
8. Resumo dos resultados
</h2>


<div class="resumo">

<ul>

<li>
<strong>Total de observações:</strong>
{total}
</li>

<li>
<strong>Transações não fraudulentas:</strong>
{nao_fraude}
</li>

<li>
<strong>Transações fraudulentas:</strong>
{fraude}
</li>

<li>
<strong>Percentual de não fraude:</strong>
{percentual_nao_fraude:.6f}%
</li>

<li>
<strong>Percentual de fraude:</strong>
{percentual_fraude:.6f}%
</li>

<li>
<strong>Razão de desbalanceamento:</strong>
{razao_desbalanceamento:.2f}:1
</li>

<li>
<strong>Imbalance Ratio:</strong>
{imbalance_ratio:.4f}
</li>

<li>
<strong>Entropia de Shannon:</strong>
{entropia:.6f} bits
</li>

</ul>

</div>


</body>

</html>
"""


# ============================================================
# 14. SALVAMENTO DO HTML
# ============================================================

CAMINHO_HTML.write_text(
    conteudo_html,
    encoding="utf-8"
)


# ============================================================
# 15. CONFIRMAÇÃO
# ============================================================

print(
    "Análise concluída com sucesso."
)

print(
    "\nHTML:"
)

print(
    CAMINHO_HTML
)

print(
    "\nPNGs:"
)

print(
    DIRETORIO_RESULTADOS
    / "target_omega_distribuicao_log.png"
)

print(
    DIRETORIO_RESULTADOS
    / "target_omega_fraudes_acumuladas.png"
)

Análise concluída com sucesso.

HTML:
/projeto_tcc_2026/resultados/analise_exploratoria_individual_features/binary_encoding/target_omega/analise_target_omega.html

PNGs:
/projeto_tcc_2026/resultados/analise_exploratoria_individual_features/binary_encoding/target_omega/target_omega_distribuicao_log.png
/projeto_tcc_2026/resultados/analise_exploratoria_individual_features/binary_encoding/target_omega/target_omega_fraudes_acumuladas.png


### <span style="color:black"> TRANS_YEAR_BE </span> ###

### <span style="color:white"> SEND_GENDER_BE </span> ###

## <span style="color:green"> FREQUÊNCY ENCODING WITH FALLBACK (1/N_TOTAL) </span> ##

### <span style="color:black"> TRANS_NUM_CARD_FEWF </span> ###

### <span style="color:white"> RECIVE_LOC_FEWF </span> ###

### <span style="color:black"> SEND_JOB_FEWF </span> ###

### <span style="color:white"> SEND_NAME_FEWF </span> ###

## <span style="color:green"> ONE-HOT ENCODING WITH IGNORE </span> ##

### <span style="color:black"> RECIVE_CATEGORY_OHEWI </span> ###

### <span style="color:white"> TRANS_WEEK_OHEWI </span> ###

## <span style="color:green"> CYCLICAL ENCODING USING SINE AND COSINE </span> ##

### <span style="color:black"> TRANS_MONTH_SEN + TRANS_MONTH_COS </span> ###

### <span style="color:white"> TRANS_HOUR_SEN + TRANS_HOUR_COS  </span> ###

## <span style="color:green"> COMUM FEATURES </span> ##

### <span style="color:black"> SEND_AGE </span> ###

### <span style="color:white"> TRANS_DAY </span> ###

### <span style="color:black"> RECIVE_LONG </span> ###

### <span style="color:white"> RECIVE_LAT </span> ###

### <span style="color:black"> SEND_POP_REGISTER </span> ###

### <span style="color:white"> SEND_LONG_REGISTER </span> ###

### <span style="color:black"> SEND_LAT_REGISTER </span> ###

### <span style="color:white"> TRANS_VALUE </span> ###